# Part 3b: redo the rel-f1 runs hit by the date bug

**Just do Runtime → Run all.** About an hour. It keeps everything that was fine, redoes only the 4 affected rel-f1 models, then downloads `final_results_part3.zip`.

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
TRIAL = f"{WORK}/results_final"
F1 = f"{WORK}/results_f1"
CACHE = "/content/cache"  # graph cache on the Colab disk, not Drive (it is large and rebuildable)
os.makedirs(F1, exist_ok=True)
F1_TASKS = "driver-dnf driver-top3 driver-position"

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

Tests (about 1–2 minutes). **Expect 26 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

Delete the 4 affected rel-f1 models

In [ ]:
BAD = ["lgbm_agg", "lgbm_agg_nohist", "rdl_sage_hist", "relgnn_hist"]
for task in F1_TASKS.split():
    for m in BAD:
        for kind in ("runs", "preds"):
            d = f"{F1}/{kind}/{task}/{m}"
            if os.path.exists(d):
                shutil.rmtree(d); print("removed", d)

Rerun whatever is missing

In [ ]:
STEPS = {
    "lgbm_table": "lgbm --features table", "lgbm_agg": "lgbm --features agg",
    "lgbm_agg_nohist": "lgbm --features agg_nohist",
    "rdl_sage": "rdl --text glove", "rdl_sage_hist": "rdl --text glove --history",
    "relgnn": "rdl --arch relgnn --text glove", "relgnn_hist": "rdl --arch relgnn --text glove --history",
}
for task in F1_TASKS.split():
    for name, cmd in STEPS.items():
        if os.path.exists(f"{F1}/runs/{task}/{name}/seed2.json"):
            print(f"done     {task:16s} {name}")
            continue
        print(f"running  {task:16s} {name}")
        !python scripts/run.py {cmd} --dataset rel-f1 --tasks {task} --seeds 0 1 2 --cache_dir {CACHE} --results {F1}

Tables, significance, download

In [ ]:
from IPython.display import Markdown, display
!python scripts/run.py tables --results {F1}
!python scripts/run.py bootstrap --dataset rel-f1 --tasks {F1_TASKS} --results {F1}
display(Markdown(open(f"{F1}/tables/main.md").read()))
display(Markdown(open(f"{F1}/tables/bootstrap.md").read()))

In [ ]:
os.makedirs("/content/part3", exist_ok=True)
shutil.copytree(F1, "/content/part3/results_f1", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("embeddings"))
shutil.copytree(f"{TRIAL}/tables", "/content/part3/results_final_tables", dirs_exist_ok=True)
shutil.make_archive("/content/final_results_part3", "zip", "/content/part3", ".")
from google.colab import files
files.download("/content/final_results_part3.zip")